## 📑 Core PreBankruptcy
 
#### 🔗 Links
- **[BRD link](https://cepalhellas.atlassian.net/wiki/x/AQAqH)**  
  Comprehensive documentation outlining the business requirements for this report.

#####Authentication

In [0]:
import boto3
from botocore.exceptions import NoCredentialsError
import traceback

"""
Accessing the Hive metastore requires specific considerations:

1. Hive metastore access is not possible via the instance profile if the cluster is on shared access mode.
   The cluster must be in shared access mode to interact with the Unity catalog.

2. To access both Unity catalog and AWS services, an access token from Amazon is required.

3. To retrieve the token secret key, access the secrets scope 'aws-databricks,' where the access key is stored per environment.
    https://docs.databricks.com/en/security/secrets/index.html

4. Once AWS credentials are obtained, clients for services like S3 and SSM can be created.

5. After you authenticate successfully querying the Hive metastore is also working.
"""

try:
    mode = dbutils.widgets.get("mode")
except Exception as e:
    mode = "shared" 

# Function to get AWS credentials from Databricks secrets.
def get_aws_credentials(secrets_scope, key_name):
    try:
        aws_secret_key = dbutils.secrets.get(scope=secrets_scope, key=key_name)
        return aws_secret_key
    except Exception as e:
        print(f"Error retrieving AWS credentials: {str(e)}")
        return None
    
# Function to get AWS credentials from Databricks secrets.
def get_scope_key_names(secrets_scope):
    try:
        # Databricks secrets configuration.
        aws_keys = dbutils.secrets.list(scope=secrets_scope)
        key_names = [key.key for key in aws_keys] if aws_keys else []
        return key_names
    
    except Exception as e:
        print(f"Error retrieving AWS keys: {str(e)}")
        return None    

secrets_scope = "aws-databricks"
aws_keys = get_scope_key_names(secrets_scope)

aws_key = aws_keys[0] if aws_keys else None

# Get AWS secret key from Databricks secrets.
aws_secret_key = get_aws_credentials(secrets_scope,aws_key)

if mode == "shared":
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)

    S3_client = boto3.client('s3', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)                           
else:
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1')

    S3_client = boto3.client('s3', region_name='eu-central-1')                           


# Function to connect to AWS and access SSM parameter.
def connect_to_aws_and_access_ssm(ssm_key):

    if aws_secret_key is None:
        print("AWS secret key not available.")
        return

    # Access SSM parameter /cfn/a360/reports_output_path.
    response = ssm_client.get_parameter(
        Name=ssm_key,
        WithDecryption=False
    )

    return response['Parameter']['Value']

# Get the Buckewt prefix from SSM to be sure.
env_prefix =connect_to_aws_and_access_ssm('/cfn/bucket_prefix')

In [0]:
from pyspark.sql.functions import coalesce, concat_ws, lit, sha2, substring, regexp_replace, array, col, explode, lit, when, split
import pandas as pd
from pyspark.sql import DataFrame
from pyspark.sql.functions import trim
import delta_table_creator, extract, transform
from delta import DeltaTable  # type: ignore
import boto3
import os

dbutils.widgets.text("run_date", '')
run_date = dbutils.widgets.get("run_date").strip()

if not run_date:
    run_date = spark.sql(f'''
                         select MAX(BusinessDate) BusinessDate from `{env_prefix}-cdh`.derivative.dexposure 
                         union
                         select MAX(BusinessDate) BusinessDate from `{env_prefix}-cdh`.landed.profits_loan_details
                         ''').collect()

    if len(run_date) != 1: raise ValueError(f"Incorrect run_date!")
    run_date = run_date[0]['BusinessDate']
    run_date = run_date.strftime('%Y-%m-%d') 

In [0]:
table_name = 'Core_PreBankruptcy'

schema = (
"BusinessDate date,"
"ExposureKey string,"
"CDI string,"
"PartyTaxRegistrationNumber string,"
"AgreementKey string,"
"ProductDescription string,"
"SubProductDescription  string,"
"SourceBucket string,"
"PacketProcessType  string,"
"Queue string,"
"Agency string,"
"ExposureStatus string,"
"ObjectiveStatusCode string,"
"AssetSegment string,"
"PartyRelationshipType string,"
"SpecialCases string,"
"LPS string,"
"EMLT string,"
"ControlArea string,"
"InsertTimeStamp timestamp"
)

In [0]:
partition_columns = ["BusinessDate"]
delivery_table_path = ""
 
# ================== Create Delta Table ==================
# env_prefix ='qa' #IL: To be Deleted!

delivery_table_path = f"`{env_prefix}-cdh-delivery`.repoint.{table_name}"
delta_table_creator.create_unity_delta_table(
    spark, delivery_table_path, partition_columns, schema
)
# if not DeltaTable.isDeltaTable(spark, delivery_table_path):
#     delta_table_creator.create_unity_delta_table(
#         spark, delivery_table_path, partition_columns, schema
#     )
  
# ================== Delete From Delta Table ==================
transform.delete_from_unity_table(
    spark,
    delivery_table_path,
    run_date,  # type: ignore
)

### Legal Action

In [0]:
casecode = spark.sql(
f'''
select distinct nc.ExposureKey ,c.casecodekey, c.portfolioname
from `{env_prefix}-cdh`.primary.casecode c
inner join  `{env_prefix}-cdh`.primary.exposure nc on c.casecodekey = nc.casecode and nc.portfolioname = 'Core Retail' and nc.businessdate = {run_date!r} AND nc.IsActiveFlag = TRUE AND nc.AssetSegment = 'SB'
where c.portfolioname = 'Core Retail'
''')
casecode.createOrReplaceTempView("casecode")

In [0]:
legalaction_core = spark.sql(
f'''
select distinct
       l.ExposureKey AccountNumber
     , l.PortfolioName
     , l.InteractionDate AS ActivityDate
     , l.InteractionTypeKey AS ActivityTypeKey
     , l.InteractionType AS ActivityName
     , l.BusinessDate
     , l.CDI
 FROM `{env_prefix}-cdh`.primary.get_interactions_by_business_date({run_date!r}) l
where 1=1
and InteractionDate <= {run_date!r}
and l.PortfolioName = 'Core Retail'
and l.SourceSystem='qca'
and InteractionTypeKey IN ('2125','2126','2127','2128','2131','2134','2135','2136','2140','2141','2144','2145','2147','2148','2153','2157','2159','2160','2161','2137','2139','2162','2146','2155','2164','2152','2156','2158','2165','2301','2302','2305','2306','2311','2312','2313','2314','2315','2321','2332','2333','2334','2335','2337','2340','2341','2343','2344','2345','2346','2347','2348','2350','2352','2356','2304','2319','2308')
 
union
 
select distinct
       c.exposurekey AccountNumber
     , l.PortfolioName
     , l.InteractionDate AS ActivityDate
     , l.InteractionTypeKey AS ActivityTypeKey
     , l.InteractionType AS ActivityName
     , l.BusinessDate
     , l.CDI
FROM `{env_prefix}-cdh`.primary.get_interactions_by_business_date({run_date!r}) l
inner join casecode c on c.casecodekey = l.casecode
where 1=1
and InteractionDate <= {run_date!r}
and l.PortfolioName = 'Core Retail'
and l.SourceSystem='qca'
and InteractionTypeKey IN ('2125','2126','2127','2128','2131','2134','2135','2136','2140','2141','2144','2145','2147','2148','2153','2157','2159','2160','2161','2137','2139','2162','2146','2155','2164','2152','2156','2158','2165','2301','2302','2305','2306','2311','2312','2313','2314','2315','2321','2332','2333','2334','2335','2337','2340','2341','2343','2344','2345','2346','2347','2348','2350','2352','2356','2304','2319','2308')
 
''')
legalaction_core.createOrReplaceTempView("legalaction_core")

### Tables from Databricks as Temps

In [0]:
ProptEliglbleStep1 = spark.sql(
f'''
SELECT AccountNumber
	 , MAX(ActivityDate) LegalActivityDate
FROM legalaction_core
WHERE ActivityTypeKey IN ('2125','2126','2127','2128','2131','2134','2135','2136','2140','2141','2144','2145','2147','2148','2153','2157','2159','2160','2161')
GROUP BY AccountNumber
''')
ProptEliglbleStep1.createOrReplaceTempView("ProptEliglbleStep1")

In [0]:
ProptNotEliglbleStep1 = spark.sql(
f'''
SELECT AccountNumber
	 , MAX(ActivityDate) LegalActivityDate
FROM legalaction_core
WHERE ActivityTypeKey IN ('2137','2139','2162','2146','2155','2164','2152','2156','2158','2165')
GROUP BY AccountNumber
''')
ProptNotEliglbleStep1.createOrReplaceTempView("ProptNotEliglbleStep1")

In [0]:
FinalEligiblePerimeter = spark.sql(
f'''
SELECT DISTINCT A.AccountNumber
FROM ProptEliglbleStep1			A
LEFT JOIN ProptNotEliglbleStep1	B	
ON A.AccountNumber = B.AccountNumber
WHERE CASE WHEN A.LegalActivityDate > (CASE WHEN B.LegalActivityDate IS NULL THEN '1900-01-01' ELSE B.LegalActivityDate END) THEN 1 ELSE 0 END = 1
''')
FinalEligiblePerimeter.createOrReplaceTempView("FinalEligiblePerimeter")

In [0]:
TotalPerimeter = spark.sql(
f'''
SELECT DISTINCT A.BusinessDate
     , A.CDI
     , A.AgreementKey
     , A.ExposureKey
     , A.PartyRelationshipType
     , A.IsActiveFlag
     , A.PortfolioName
     , A.PartyTaxRegistrationNumber
     , CASE WHEN C.AccountNumber IS NOT NULL THEN 1 ELSE 0 END ProptwxeytikoFlag
     , A.AssetSegment
     , LPS_TBL.OperationalLegalStatusLevel3 AS LPS
     , A.SourceSystem
     , A.ProductKey
     , A.Bucket
     , A.CaseCode
     , A.ExposureStatus
FROM `{env_prefix}-cdh`.primary.exposure           A
LEFT JOIN `{env_prefix}-cdh`.derivative.dlegalprocessstatusagreementdebtor LPS_TBL
ON LPS_TBL.AgreementKey = A.AgreementKey AND LPS_TBL.CDI = A.CDI AND LPS_TBL.BusinessDate = A.BusinessDate
LEFT JOIN FinalEligiblePerimeter	C	
ON A.ExposureKey = C.AccountNumber

-- FROM `{env_prefix}-cdh`.primary.exposure           A
-- LEFT JOIN FinalEligiblePerimeter	C	
-- ON A.ExposureKey = C.AccountNumber
WHERE 1=1
AND A.IsActiveFlag = TRUE
AND A.AssetSegment = 'SB'
AND A.BusinessDate = {run_date!r}
AND A.PortfolioName = 'Core Retail'
''')
TotalPerimeter.createOrReplaceTempView("TotalPerimeter")

In [0]:
EligibleDebtor = spark.sql(
f'''
SELECT DISTINCT 
		CDI
	  ,	PartyTaxRegistrationNumber 
FROM TotalPerimeter 
WHERE ProptwxeytikoFlag = 1
''')
EligibleDebtor.createOrReplaceTempView("EligibleDebtor")

In [0]:
PreFinalStep1 = spark.sql(
f'''
SELECT DISTINCT 
		A.*
FROM
(
    SELECT A.* 
	FROM TotalPerimeter			A 
	INNER JOIN EligibleDebtor	B 
 	ON A.CDI = B.CDI
		
UNION

	SELECT A.* 
	FROM TotalPerimeter			A 
	INNER JOIN EligibleDebtor	B 
 	ON A.PartyTaxRegistrationNumber = B.PartyTaxRegistrationNumber
	WHERE A.PartyTaxRegistrationNumber <> '000000000'
  	AND A.PartyTaxRegistrationNumber <> ''
) A
''')
PreFinalStep1.createOrReplaceTempView("PreFinalStep1")

In [0]:
PtwEliglbleStep1 = spark.sql(
f'''
SELECT AccountNumber
	 , MAX(ActivityDate) LegalActivityDate
FROM legalaction_core
WHERE ActivityTypeKey IN ('2301','2302','2305','2306','2311','2312','2313','2314','2315','2321','2332','2333','2334','2335','2337','2340','2341','2343','2344','2345','2346','2347','2348','2350','2352','2356')
GROUP BY AccountNumber
''')
PtwEliglbleStep1.createOrReplaceTempView("PtwEliglbleStep1")

In [0]:
PtwNotEliglbleStep1 = spark.sql(
f'''
SELECT AccountNumber
	 , MAX(ActivityDate) LegalActivityDate
FROM legalaction_core
WHERE ActivityTypeKey IN ('2304','2319','2308')
GROUP BY AccountNumber
''')
PtwNotEliglbleStep1.createOrReplaceTempView("PtwNotEliglbleStep1")

In [0]:
FinalEligiblePerimeterPtw = spark.sql(
f'''
SELECT DISTINCT A.AccountNumber
FROM PtwEliglbleStep1			A
LEFT JOIN PtwNotEliglbleStep1	B	
ON A.AccountNumber = B.AccountNumber
WHERE CASE WHEN A.LegalActivityDate > (CASE WHEN B.LegalActivityDate IS NULL THEN '1900-01-01' ELSE B.LegalActivityDate END) THEN 1 ELSE 0 END = 1
''')
FinalEligiblePerimeterPtw.createOrReplaceTempView("FinalEligiblePerimeterPtw")

In [0]:
TotalPerimeterPtw = spark.sql(
f'''
SELECT DISTINCT A.BusinessDate
     , A.CDI
     , A.AgreementKey
     , A.ExposureKey
     , A.PartyRelationshipType
     , A.IsActiveFlag
     , A.PortfolioName
     , A.PartyTaxRegistrationNumber
     , CASE WHEN C.AccountNumber IS NOT NULL THEN 1 ELSE 0 END ProptwxeytikoFlag
     , A.AssetSegment
     , LPS_TBL.OperationalLegalStatusLevel3 AS LPS
     , A.SourceSystem
     , A.ProductKey
     , A.Bucket
     , A.CaseCode
     , A.ExposureStatus
FROM `{env_prefix}-cdh`.primary.exposure           A
LEFT JOIN `{env_prefix}-cdh`.derivative.dlegalprocessstatusagreementdebtor LPS_TBL
ON LPS_TBL.AgreementKey = A.AgreementKey AND LPS_TBL.CDI = A.CDI AND LPS_TBL.BusinessDate = A.BusinessDate
LEFT JOIN FinalEligiblePerimeterPtw	C	
ON A.ExposureKey = C.AccountNumber
WHERE 1=1
AND A.IsActiveFlag = TRUE
AND A.AssetSegment = 'SB'
AND A.BusinessDate = {run_date!r}
AND A.PortfolioName = 'Core Retail'
''')
TotalPerimeterPtw.createOrReplaceTempView("TotalPerimeterPtw")

In [0]:
EligibleDebtorPtw = spark.sql(
f'''
SELECT DISTINCT 
		CDI
	  ,	PartyTaxRegistrationNumber 
FROM TotalPerimeterPtw 
WHERE ProptwxeytikoFlag = 1
''')
EligibleDebtorPtw.createOrReplaceTempView("EligibleDebtorPtw")

In [0]:
PreFinalStep1Ptw = spark.sql(
f'''
SELECT DISTINCT 
			A.*
FROM
(
 SELECT A.* 
 FROM TotalPerimeterPtw				A 
 INNER JOIN EligibleDebtorPtw		B 
 ON A.CDI = B.CDI

 UNION
			
 SELECT A.*
 FROM TotalPerimeterPtw			  A 
 INNER JOIN EligibleDebtorPtw	  B 
 ON A.PartyTaxRegistrationNumber = B.PartyTaxRegistrationNumber
 WHERE A.PartyTaxRegistrationNumber <> '000000000'
 AND A.PartyTaxRegistrationNumber <> ''
) A
''')
PreFinalStep1Ptw.createOrReplaceTempView("PreFinalStep1Ptw")

##### final_df

In [0]:
final_df = spark.sql(
f'''
SELECT DISTINCT 	
	  CAST({run_date!r} AS date)                                                                              AS BusinessDate
  ,	CAST(A.ExposureKey AS STRING)                                                                           AS ExposureKey
  ,	CAST(A.CDI AS STRING)                                                                                   AS CDI
  ,	CAST(A.PartyTaxRegistrationNumber AS STRING)                                                            AS PartyTaxRegistrationNumber
  ,	CAST(A.AgreementKey AS STRING)                                                                          AS AgreementKey
  , CAST(P.SubProduct AS STRING) 	 			                                                                    AS ProductDescription
  , CAST(P.BankingUnitCode AS STRING) 				                                                              AS SubProductDescription         
  ,	CAST(A.Bucket AS STRING)                                                                                AS SourceBucket
  ,	CAST(L.PACKET_PROCESS_TYPE AS STRING)                                                                   AS PacketProcessType
  ,	CAST(C.Queue AS STRING)                                                                                 AS Queue
  ,	CAST(C.Agency AS STRING)                                                                                AS Agency
  ,	CAST(A.ExposureStatus AS STRING)                                                                        AS ExposureStatus
  ,	CAST(D.ObjectiveStatusCode AS STRING)                                                                   AS ObjectiveStatusCode
  ,	CAST(A.AssetSegment AS STRING)                                                                          AS AssetSegment
  ,	CAST(A.PartyRelationshipType AS STRING)                                                                 AS PartyRelationshipType
  ,	CAST(CASE WHEN C.WatchlistFlag = 'Υπαναχώρηση ΕΤΕΑΝ' THEN C.WatchlistFlag ELSE NULL END AS STRING)      AS SpecialCases
  , CAST(A.LPS AS STRING)                                                                                   AS LPS
  , CAST(CASE WHEN P.SubProduct = 'ΒΟΗΘΗΤΙΚΟΙ ΛΟΓΑΡΙΑΣΜΟΙ' THEN 'Yes' ELSE NULL END AS STRING)              AS EMLT
  , CAST('Bank' AS STRING)                                                                                  AS ControlArea
  , CAST(getdate() AS TIMESTAMP)                                                                            AS InsertTimeStamp
FROM PreFinalStep1						 A
LEFT JOIN `{env_prefix}-cdh`.primary.product P
ON P.ProductKey = A.ProductKey AND P.BusinessDate = A.BusinessDate AND P.SourceSystem = A.SourceSystem
LEFT JOIN `{env_prefix}-cdh`.primary.casecode 	C 
ON C.CaseCodeKey = A.CaseCode AND C.BusinessDate = A.BusinessDate AND C.PortfolioName = 'Core Retail'
LEFT JOIN `{env_prefix}-cdh`.derivative.dexposure D
ON D.ExposureKeyQCAlpha = A.ExposureKey AND D.BusinessDate = A.BusinessDate
LEFT JOIN `{env_prefix}-cdh-delivery`.repoint.core_retailprimary L 
ON L.ACCOUNT_NUMBER = A.ExposureKey AND L.BusinessDate = {run_date!r}
LEFT JOIN PreFinalStep1Ptw F
ON F.ExposureKey = A.ExposureKey
WHERE A.PartyRelationshipType = 'Primary Owner'   
AND	CASE WHEN F.ExposureKey IS NOT NULL THEN 1 ELSE 0 END = 0
''')
final_df.createOrReplaceTempView("final_df")

In [0]:
if final_df.rdd.isEmpty():
    raise ValueError(f"Core_PreBankruptcy: final DataFrame is empty - aborting write to {delivery_table_path} to avoid overwriting with no data")
final_df.write \
    .format("delta") \
    .partitionBy("BusinessDate") \
    .mode("append") \
    .saveAsTable(delivery_table_path) # in order to write in delta_lake

In [0]:
%python
from pyspark.sql.functions import col, count, concat

def check_table(delivery_table_path, run_date, key_columns, duplicate_check_action="error", empty_check_action="error"):
    errors = []
    warnings = []

    if key_columns:
        # Create the composite key column if there are multiple key columns
        if len(key_columns) > 1:
            composite_key = concat(*[col(c) for c in key_columns]).alias("CompositeKey")
            key_column_name = "CompositeKey"
            key_columns_string = ", ".join(key_columns)
        else:
            key_column_name = key_columns[0]
            key_columns_string = key_columns[0]

        # Duplicate Checks
        duplicate_count = (
            spark.table(delivery_table_path)
            .filter(col("BusinessDate") == run_date)
            .withColumn("CompositeKey", composite_key if len(key_columns) > 1 else col(key_column_name))
            .groupBy(key_column_name)
            .count()
            .filter("count > 1")
            .select(key_column_name, "count")
            .toPandas()
            .to_dict(orient='records')
        )

        if duplicate_count:
            message = f"Duplicate {key_columns_string} entries found: {duplicate_count}"
            if duplicate_check_action == "error":
                errors.append(message)
            elif duplicate_check_action == "warning":
                warnings.append(message)

    # Check if the table for the specific date is empty
    row_count = spark.sql(f"""
        SELECT * FROM {delivery_table_path}
        WHERE BusinessDate = '{run_date}'
    """).count()

    if row_count == 0:
        message = f"The {delivery_table_path} dataset is empty for the date {run_date}."
        if empty_check_action == "error":
            errors.append(message)
        elif empty_check_action == "warning":
            warnings.append(message)

    if warnings:
        print("Warnings:\n" + "\n".join(warnings))

    if errors:
        raise ValueError("Errors found:\n" + "\n".join(errors))

key_columns = ["ExposureKey"]
check_table(delivery_table_path, run_date, key_columns, duplicate_check_action="error", empty_check_action="warning")

In [0]:
import re
def rename_and_cleanup_files(output_path, file_name):
    part_file_regex = re.compile(r'part-.*')
    system_file_regex = re.compile(r'^_(committed_|started_|SUCCESS)')
 
    # Move the part file to the final filename.
    files = dbutils.fs.ls(output_path)
    for file in files:
        print(f"file: {file.name}")
        if part_file_regex.match(file.name):
            original_path = file.path
            new_path = output_path + file_name
            dbutils.fs.mv(original_path, new_path)
 
    # Remove system files from the common folder.
    files = dbutils.fs.ls(output_path)
    for file in files:
        if system_file_regex.match(file.name):
            dbutils.fs.rm(file.path, recurse=False)
 
# Example usage:
# rename_and_cleanup_files(output_path, file_name)

In [0]:
PreBankruptcy = spark.sql(
f'''
select * from {delivery_table_path}
where BusinessDate = {run_date!r}
'''
)
PreBankruptcy.createOrReplaceTempView("PreBankruptcy")

In [0]:
output_path = "/Volumes/"+env_prefix+"-cdh-delivery/repoint/exports/Core/PreBankruptcy/"
file_name = "PreBankruptcy_" +run_date+ ".csv"
PreBankruptcy.coalesce(1).write.mode("append").option("header", True).csv(f"{output_path}")
rename_and_cleanup_files(output_path, file_name)